# 05 · Comparação entre bases — o diferencial do trabalho

Cinco fontes externas, cada uma respondendo uma pergunta que o arquivo entregue
não responde sozinho.

| fonte | o que resolve |
|---|---|
| **BRFSS 2015 original** | mede o viés do próprio arquivo entregue |
| **Vigitel 2015** | os fatores valem no Brasil? |
| **NHANES** (prior) | quanta doença fica sem diagnóstico |
| **CDC Open Data** | valida nossa estimativa contra o número oficial |
| **Painel Medicaid** | acesso causa diagnóstico? |

> Documentos: [`docs/09`](../docs/09-comparacao-binacional.md), [`docs/12`](../docs/12-frente2-positive-unlabeled.md), [`docs/14`](../docs/14-frente4-medicaid-experimento-natural.md)


In [1]:
import sys, json
from pathlib import Path

# a raiz e onde existe src/ — funciona rodando de notebooks/ ou da raiz do repo
RAIZ = Path.cwd()
if not (RAIZ / "src").exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))

import numpy as np, pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)

GOLD = RAIZ / "data" / "processed" / "gold"
def ler(nome, base=GOLD):
    return json.loads((base / nome).read_text(encoding="utf-8"))


## Brasil × EUA — mesmo ano, mesmo desenho, mesmo modelo


In [2]:
bi = ler("_comparacao_binacional.json", RAIZ / "data" / "external" / "vigitel")
pd.DataFrame(bi["prevalencia"]).T


,n,n_efetivo,bruta_%,ponderada_%
Brasil_Vigitel_2015,54064.0,10843.0,9.707,7.076
EUA_BRFSS_2015,440658.0,109019.0,12.993,10.500


In [3]:
o = pd.DataFrame(bi["odds_ratio"]).T
o[["OR_Brasil", "IC_Brasil", "OR_EUA", "IC_EUA", "razao_BR_EUA", "ic_sobrepoe"]]


,OR_Brasil,IC_Brasil,OR_EUA,IC_EUA,razao_BR_EUA,ic_sobrepoe
frutas,1.299,[1.10; 1.53],0.898,[0.86; 0.94],1.45,False
fumante,1.268,[1.08; 1.49],1.149,[1.10; 1.20],1.1,True
sexo,1.273,[1.08; 1.50],1.175,[1.12; 1.23],1.08,True
atividade_fisica,0.846,[0.72; 1.00],0.795,[0.76; 0.84],1.06,True
idade_faixa,1.234,[1.20; 1.27],1.24,[1.23; 1.25],1.0,True
hipertensao,3.136,[2.64; 3.73],3.146,[2.99; 3.31],1.0,True
escolaridade3,0.732,[0.65; 0.82],0.768,[0.74; 0.80],0.95,True
imc5,1.228,[1.15; 1.31],1.454,[1.43; 1.48],0.84,False


**Seis de oito fatores convergem.** `hipertensao` (3,136 vs 3,146) e `idade_faixa`
(1,234 vs 1,240) coincidem na **terceira casa decimal**, em dois países e dois
sistemas de saúde. É a evidência mais forte do trabalho de que os fatores centrais
são robustos e **transferem**.

**Duas divergências reais:**
- `frutas` **inverte de direção** (BR 1,30 · EUA 0,90)
- **o IMC pesa 16% menos no Brasil** (1,23 vs 1,45 por 5 kg/m²) — um escore
  calibrado nos EUA **superestima** o IMC aqui


## Quanta doença está escondida


In [4]:
pu = ler("_frente2_pu.json")
bbe = pu["bbe"]
print("c identificavel so com os dados?", bbe["identificado"])
print("fracao rotulada no topo, por resolucao:", bbe["sensibilidade"])
print("espalhamento", bbe["espalhamento_na_grade"], "> limite", bbe["limite_de_plato"])
print()
print(f"c usado (premissa exogena, NHANES): {pu['premissa']['c_nhanes']}")
display(pd.DataFrame(pu["sensibilidade_a_c"]))


c identificavel so com os dados? False
fracao rotulada no topo, por resolucao: {'q=25': 0.6218, 'q=50': 0.6861, 'q=100': 0.7402}
espalhamento 0.1184 > limite 0.0298

c usado (premissa exogena, NHANES): 0.724


,c,subdiagnostico_%,prev_verdadeira_%,prev_diagnosticada_%,ocultos_pp,ocultos_na_amostra
0,0.650,35.0,15.816,10.671,5.145,58204
1,0.700,30.0,14.734,10.671,4.063,54107
2,0.724,27.6,14.258,10.671,3.587,52328
3,0.780,22.0,13.249,10.671,2.578,48585
4,0.850,15.0,12.161,10.671,1.490,44586
5,0.900,10.0,11.486,10.671,0.815,42109


**O estimador se recusa a estimar, e isso e um resultado.** A fracao rotulada no
topo depende da resolucao do grid (0,62 a 0,74), logo **nao existe regiao pura de
positivos** no espaco de 60 perguntas — nenhum perfil construivel a partir delas
isola um subgrupo em que todos tenham diabetes. E o teto de informacao do
questionario, medido por uma segunda via.

`c` fica como premissa exogena do NHANES (0,724), sempre com a faixa de
sensibilidade acima. Ver a retratacao em `docs/12` Passo 1.

E a ironia que fecha o notebook 02:

```
13,93%   arquivo entregue, sem peso         (seleção, para cima)
10,67%   BRFSS ponderado, diagnóstico
14,29%   prevalência VERDADEIRA estimada    (subdiagnóstico, para baixo)
```

Quem usasse o arquivo cru chegaria a 13,93% — perto dos 14,29% corretos,
**pelo motivo errado**, por dois vieses de sinal oposto.


## Quem são os prováveis não diagnosticados


In [5]:
pd.DataFrame(pu["perfil"]).T[
    ["n", "idade_media", "imc_medio", "%_hipertensao",
     "%_check_up_no_ano", "%_sem_consulta_por_custo", "%_minoria"]]


,n,idade_media,imc_medio,%_hipertensao,%_check_up_no_ano,%_sem_consulta_por_custo,%_minoria
provaveis_ocultos,20000.0,62.4,32.3,75.8,29.9,23.1,34.5
diagnosticados,57256.0,64.4,31.7,74.9,88.1,11.1,29.4
demais_nao_rotulados,355712.0,53.4,27.2,32.3,73.7,8.8,22.2


**Clinicamente iguais aos diagnosticados** — hipertensão 74,7% contra 74,9%,
IMC 32,3 contra 31,7 — e com o **acesso dos excluídos**: um terço do check-up,
o dobro de renúncia a consulta por custo, mais minorias.

É exatamente a população que um programa de rastreamento deveria alcançar, e a
que um classificador supervisionado ingênuo **ignora por construção**.


## Acesso causa diagnóstico? O experimento natural


In [6]:
med = ler("_frente4_medicaid.json", RAIZ / "data" / "external" / "medicaid")
display(pd.DataFrame(med["did_baixa_renda"])[
    ["desfecho", "efeito_pp", "ic95", "p"]])
print("\nPlacebo (renda alta, não elegível):")
display(pd.DataFrame(med["placebo_renda_alta"])[["desfecho", "efeito_pp", "p"]])


,desfecho,efeito_pp,ic95,p
0,diabetes,-0.3969,"[-1.0301, 0.2364]",0.2193
1,cobertura,3.1133,"[0.0203, 6.2062]",0.0485
2,barreira_custo,-2.2429,"[-4.4853, -0.0006]",0.0499
3,colesterol_checado,1.0819,"[-0.725, 2.8888]",0.2406



Placebo (renda alta, não elegível):


,desfecho,efeito_pp,p
0,diabetes,0.0805,0.6982
1,cobertura,0.4797,0.0869
2,barreira_custo,-0.2017,0.5013
3,colesterol_checado,0.3217,0.5556


In [7]:
poder = med["poder_do_desenho"]
for k in ["efeito_MAXIMO_esperado_sobre_diagnostico_pp",
          "diferenca_minima_detectavel_pp", "razao_mde_sobre_efeito_esperado"]:
    print(f"{k:48} {poder[k]}")
print(f"\n{poder['veredito']}")


efeito_MAXIMO_esperado_sobre_diagnostico_pp      0.16
diferenca_minima_detectavel_pp                   0.9047
razao_mde_sobre_efeito_esperado                  5.7

o desenho NAO tem poder para detectar o efeito esperado — o nulo e inconclusivo, nao evidencia de ausencia


A expansão do Medicaid **aumentou o acesso** (+3,11 p.p. de cobertura) mas o efeito
sobre o diagnóstico é nulo. Antes de interpretar, calculamos o poder:

**Efeito máximo plausível 0,16 p.p. contra diferença mínima detectável de 0,90 p.p.**

O desenho não podia detectar o efeito esperado. **"Não detectamos" não é "não
existe"** — e sem esse cálculo o nulo seria lido como a conclusão oposta à
evidência do projeto.
